# SPI Service Log Analysis

This notebook reads the `spi-service.log` file, extracts data from "Sending to client:" lines,
identifies checkpoints (marked by `[-99999, sequence, 0]`), and creates a summary DataFrame.

In [ ]:
import json
import re
import pandas as pd
from typing import List, Tuple

## Step 1: Read and Parse the Log File

In [ ]:
# Read the log file
log_file_path = 'spi-service.log'

with open(log_file_path, 'r') as f:
    log_content = f.read()

print(f"Log file loaded. Total characters: {len(log_content)}")

In [ ]:
# Extract all "Sending to client:" lines and parse JSON
# Pattern: Sending to client: '{...JSON...}'
pattern = r"Sending to client: '(\{.*?\})'"

matches = re.findall(pattern, log_content, re.DOTALL)
print(f"Found {len(matches)} 'Sending to client:' entries")

## Step 2: Extract Data Arrays and Combine

In [ ]:
# Parse JSON and extract data arrays
all_data_points: List[List[int]] = []
data_entry_count = 0

for match in matches:
    try:
        json_obj = json.loads(match)
        # Only process entries with "type": "data"
        if json_obj.get('type') == 'data' and 'data' in json_obj:
            data_array = json_obj['data']
            all_data_points.extend(data_array)
            data_entry_count += 1
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        continue

print(f"Processed {data_entry_count} data entries")
print(f"Total data points collected: {len(all_data_points)}")

In [ ]:
# Show first few data points
print("First 5 data points:")
for i, dp in enumerate(all_data_points[:5]):
    print(f"  [{i}]: {dp}")

print("\nLast 5 data points:")
for i, dp in enumerate(all_data_points[-5:]):
    print(f"  [{len(all_data_points)-5+i}]: {dp}")

## Step 3: Identify Checkpoints

Checkpoints are identified by the pattern `[-99999, sequence_number, 0]`

In [ ]:
# Find all checkpoints
CHECKPOINT_MARKER = -99999

checkpoints: List[Tuple[int, int]] = []  # (index, sequence_number)

for idx, data_point in enumerate(all_data_points):
    if len(data_point) >= 1 and data_point[0] == CHECKPOINT_MARKER:
        sequence_num = data_point[1] if len(data_point) > 1 else None
        checkpoints.append((idx, sequence_num))
        print(f"Checkpoint found at index {idx}: sequence = {sequence_num}, data = {data_point}")

print(f"\nTotal checkpoints found: {len(checkpoints)}")

## Step 4: Create Summary DataFrame

In [ ]:
# Build the checkpoint summary DataFrame
summary_data = []

# Handle data before first checkpoint
if checkpoints:
    first_checkpoint_idx = checkpoints[0][0]
    if first_checkpoint_idx > 0:
        summary_data.append({
            'segment': 'Start -> Checkpoint 1',
            'from_checkpoint': 'Start',
            'to_checkpoint': checkpoints[0][1],
            'from_index': 0,
            'to_index': first_checkpoint_idx - 1,
            'data_points_count': first_checkpoint_idx
        })

# Handle segments between checkpoints
for i in range(len(checkpoints) - 1):
    current_cp_idx, current_cp_seq = checkpoints[i]
    next_cp_idx, next_cp_seq = checkpoints[i + 1]
    
    # Data points between checkpoints (excluding the checkpoint markers)
    start_idx = current_cp_idx + 1
    end_idx = next_cp_idx - 1
    count = next_cp_idx - current_cp_idx - 1
    
    summary_data.append({
        'segment': f'Checkpoint {current_cp_seq} -> Checkpoint {next_cp_seq}',
        'from_checkpoint': current_cp_seq,
        'to_checkpoint': next_cp_seq,
        'from_index': start_idx,
        'to_index': end_idx,
        'data_points_count': count
    })

# Handle data after last checkpoint
if checkpoints:
    last_checkpoint_idx = checkpoints[-1][0]
    last_checkpoint_seq = checkpoints[-1][1]
    remaining_count = len(all_data_points) - last_checkpoint_idx - 1
    if remaining_count > 0:
        summary_data.append({
            'segment': f'Checkpoint {last_checkpoint_seq} -> End',
            'from_checkpoint': last_checkpoint_seq,
            'to_checkpoint': 'End',
            'from_index': last_checkpoint_idx + 1,
            'to_index': len(all_data_points) - 1,
            'data_points_count': remaining_count
        })

# Create DataFrame
df_summary = pd.DataFrame(summary_data)
print("Checkpoint Summary DataFrame:")
df_summary

In [ ]:
# Display summary statistics
print("\n=== Summary Statistics ===")
print(f"Total data points (including checkpoints): {len(all_data_points)}")
print(f"Total checkpoints: {len(checkpoints)}")
print(f"Total data points (excluding checkpoints): {len(all_data_points) - len(checkpoints)}")

if not df_summary.empty:
    print(f"\nData points per segment:")
    print(f"  Min: {df_summary['data_points_count'].min()}")
    print(f"  Max: {df_summary['data_points_count'].max()}")
    print(f"  Mean: {df_summary['data_points_count'].mean():.2f}")
    print(f"  Total: {df_summary['data_points_count'].sum()}")

## Step 5: Detailed Checkpoint List

In [ ]:
# Create a DataFrame with checkpoint details
checkpoint_details = []
for idx, seq in checkpoints:
    checkpoint_details.append({
        'checkpoint_sequence': seq,
        'array_index': idx,
        'raw_data': all_data_points[idx]
    })

df_checkpoints = pd.DataFrame(checkpoint_details)
print("Checkpoint Details:")
df_checkpoints

## Step 6: Create Combined Data Array (Excluding Checkpoints)

In [ ]:
# Create a clean data array without checkpoint markers
checkpoint_indices = set(idx for idx, _ in checkpoints)
clean_data = [dp for i, dp in enumerate(all_data_points) if i not in checkpoint_indices]

print(f"Clean data array length: {len(clean_data)}")
print(f"\nFirst 5 clean data points:")
for i, dp in enumerate(clean_data[:5]):
    print(f"  [{i}]: {dp}")

In [ ]:
# Convert to DataFrame for further analysis
df_data = pd.DataFrame(clean_data, columns=['Channel_1', 'Channel_2', 'Channel_3'])
print("Data as DataFrame (first 10 rows):")
df_data.head(10)

In [ ]:
# Basic statistics for the data channels
print("\nData Channel Statistics:")
df_data.describe()